# Module 10: Data Architecture & Reusable ML Assets

**Aligned with Techcombank JD**: Data Architecture, Data Integration, ML Data Assets

## Topics
1. Medallion Architecture (Bronze → Silver → Gold)
2. Building reusable data assets for Data Scientists
3. Feature Store patterns
4. Data product mindset
5. Multi-source integration patterns

In [ ]:
# ── SparkSession: Databricks Connect (remote) / Local fallback ──
from pathlib import Path

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()
    MODE = 'databricks'
    S3_RAW = "s3a://sparkling-data-test/data/raw"
    print(f"✅ Databricks Connect | Spark {spark.version}")
except Exception:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName("Module10-DataArchitecture").master("local[*]").config("spark.sql.shuffle.partitions", "8").getOrCreate()
    MODE = 'local'
    S3_RAW = None
    print(f"✅ Local Spark {spark.version} | UI: http://localhost:4040")

DATA_RAW = Path("../data/raw")  # local CSV fallback path
print(f"Mode: {MODE}")

---
## 1. Medallion Architecture

**Standard for modern data platforms** (Databricks, AWS Lake Formation)

```
┌─────────────┐    ┌─────────────┐    ┌─────────────┐
│   BRONZE    │ →  │   SILVER    │ →  │    GOLD     │
│  Raw Data   │    │  Cleansed   │    │  Business   │
│  As-Is      │    │  Validated  │    │  Aggregated │
└─────────────┘    └─────────────┘    └─────────────┘
```

| Layer | Purpose | Schema | Updates |
|-------|---------|--------|--------|
| Bronze | Raw ingestion | Schema-on-read | Append-only |
| Silver | Cleansed, validated | Enforced schema | SCD/Merge |
| Gold | Business aggregates | Denormalized | Overwrite/Merge |

In [ ]:
# ── Load data (S3 Parquet or local CSV) ──
transactions_bronze = spark.read.parquet(f"{S3_RAW}/transactions") if MODE == "databricks" else spark.read.csv(str(DATA_RAW / "transactions.csv"), header=True, inferSchema=True)
customers_bronze = spark.read.parquet(f"{S3_RAW}/customers") if MODE == "databricks" else spark.read.csv(str(DATA_RAW / "customers.csv"), header=True, inferSchema=True)

transactions_bronze = transactions_bronze.withColumn("_ingested_at", current_timestamp()).withColumn("_source", lit("core_banking"))
print(f"Bronze transactions: {transactions_bronze.count():,}")

In [ ]:
# === SILVER LAYER: Cleansed, validated ===
transactions_silver = transactions_bronze.filter(
    (col("amount") > 0) & 
    (col("status").isin(["Completed", "Pending", "Failed", "Reversed"]))
).withColumn(
    "txn_date", to_date(col("txn_datetime"))
).withColumn(
    "txn_hour", hour(col("txn_datetime"))
).withColumn(
    "is_business_hours", (col("txn_hour").between(8, 17))
)

print(f"Silver transactions: {transactions_silver.count():,}")
transactions_silver.select("txn_id", "txn_date", "txn_hour", "is_business_hours", "amount").show(5)

In [ ]:
# === GOLD LAYER: Business aggregates ===
# Daily transaction summary - ready for BI dashboards
daily_summary_gold = transactions_silver.groupBy("txn_date", "channel").agg(
    count("txn_id").alias("txn_count"),
    sum("amount").alias("total_amount"),
    avg("amount").alias("avg_amount"),
    countDistinct("account_id").alias("unique_accounts")
)

daily_summary_gold.orderBy("txn_date", "channel").show(10)

---
## 2. Reusable Data Assets for ML

**JD Requirement**: *"Create re-usable data assets to enable data scientists to build and deploy ML models faster"*

### Feature Engineering Patterns

In [ ]:
# ── Load data (S3 Parquet or local CSV) ──
accounts = spark.read.parquet(f"{S3_RAW}/accounts") if MODE == "databricks" else spark.read.csv(str(DATA_RAW / "accounts.csv"), header=True, inferSchema=True)

txn_with_customer = transactions_silver.join(
    accounts.select("account_id", "customer_id"), "account_id"
)
customer_features = txn_with_customer.groupBy("customer_id").agg(
    count("txn_id").alias("total_txn_count"),
    sum("amount").alias("total_txn_amount"),
    avg("amount").alias("avg_txn_amount"),
    stddev("amount").alias("std_txn_amount"),
    sum(when(col("channel") == "Mobile App", 1).otherwise(0)).alias("mobile_txn_count"),
    sum(when(col("channel") == "Branch", 1).otherwise(0)).alias("branch_txn_count"),
    sum(when(col("is_business_hours"), 1).otherwise(0)).alias("business_hours_txn"),
    countDistinct("txn_date").alias("active_days"),
    sum(when(col("status") == "Failed", 1).otherwise(0)).alias("failed_txn_count"),
    max("amount").alias("max_txn_amount")
)
customer_features = customer_features.withColumn(
    "mobile_preference_ratio", col("mobile_txn_count") / col("total_txn_count")
).withColumn(
    "failure_rate", col("failed_txn_count") / col("total_txn_count")
).withColumn(
    "txn_frequency", col("total_txn_count") / col("active_days")
)
customer_features.show(5)

In [ ]:
# Save as reusable feature table
feature_path = DATA_PROCESSED / "features" / "customer_features"
feature_path.parent.mkdir(parents=True, exist_ok=True)
customer_features.write.mode("overwrite").parquet(str(feature_path))
print(f"✅ Customer features saved: {feature_path}")

---
## 3. Feature Store Pattern

**For Data Scientists to easily discover and use features**

In [ ]:
def register_feature_table(df, feature_name, primary_key, description, version="1.0"):
    """
    Pattern for registering features with metadata.
    In production: Use MLflow, Feast, or Databricks Feature Store.
    """
    metadata = {
        "feature_name": feature_name,
        "primary_key": primary_key,
        "columns": df.columns,
        "row_count": df.count(),
        "description": description,
        "version": version,
        "created_at": str(current_timestamp())
    }
    print(f"\n📊 Feature Table: {feature_name}")
    print(f"   Primary Key: {primary_key}")
    print(f"   Columns: {len(df.columns)}")
    print(f"   Rows: {metadata['row_count']:,}")
    print(f"   Description: {description}")
    return metadata

# Register our feature table
register_feature_table(
    customer_features, 
    "customer_transaction_features",
    "customer_id",
    "Customer transaction behavior features for churn prediction and segmentation models"
)

---
## 4. Multi-Source Integration

**JD Requirement**: *"Obtain and integrate data from various sources"*

In [ ]:
def integrate_multi_source(spark, sources_config):
    """
    Pattern for integrating multiple data sources.
    
    sources_config example:
    {
        "core_banking": {"type": "jdbc", "url": "...", "table": "transactions"},
        "crm": {"type": "parquet", "path": "s3://..."},
        "external_api": {"type": "json", "path": "..."},
    }
    """
    datasets = {}
    
    for name, config in sources_config.items():
        source_type = config.get("type")
        
        if source_type == "csv":
            df = spark.read.csv(config["path"], header=True, inferSchema=True)
        elif source_type == "parquet":
            df = spark.read.parquet(config["path"])
        elif source_type == "jdbc":
            df = spark.read.jdbc(config["url"], config["table"], properties=config.get("properties", {}))
        elif source_type == "delta":
            df = spark.read.format("delta").load(config["path"])
        else:
            raise ValueError(f"Unknown source type: {source_type}")
        
        # Add source metadata
        df = df.withColumn("_source", lit(name)).withColumn("_ingested_at", current_timestamp())
        datasets[name] = df
        print(f"✅ Loaded {name}: {df.count():,} rows")
    
    return datasets

# Example usage
sources = {
    "transactions": {"type": "csv", "path": str(DATA_RAW / "transactions.csv")},
    "customers": {"type": "csv", "path": str(DATA_RAW / "customers.csv")},
    "accounts": {"type": "csv", "path": str(DATA_RAW / "accounts.csv")},
}

datasets = integrate_multi_source(spark, sources)

---
## 5. Data Product Checklist

**JD Requirement**: *"Drive delivery of data products and services"*

### Before releasing a data product:

✅ **Documentation**
- [ ] Schema documented with column descriptions
- [ ] Data lineage tracked (source → transformations → output)
- [ ] SLA defined (freshness, completeness)

✅ **Quality**
- [ ] Data quality checks automated
- [ ] Anomaly detection in place
- [ ] Historical validation passed

✅ **Compliance**
- [ ] PII identified and masked/encrypted
- [ ] Access controls configured
- [ ] Audit logging enabled

✅ **Performance**
- [ ] Partitioning strategy defined
- [ ] Query patterns optimized
- [ ] Caching configured for hot data

In [ ]:
spark.stop()